In [2]:
from abc import ABC
from typing import Union
from cudakima import AkimaInterpolant1D
import numpy as np
try:
    import cupy as xp
except (ImportError, ModuleNotFoundError):
    import numpy as xp

import jax.numpy as jnp

In [ ]:
class SplinedModel(ABC):
    def __init__(self, 
                 use_gpu: bool =False, 
                 ftol: float = 1e-2,
                 interp_kwargs: dict = {}
                 ):
        self.xp = xp if use_gpu else np
        self.ftol = ftol
        self.interp = AkimaInterpolant1D(use_gpu=use_gpu, **interp_kwargs)
    
    def base_model(self, *args, **kwargs) -> Union[xp.ndarray, jnp.ndarray]:
        """
        This method should be implemented by subclasses to define the base model. Shape of the otput should be (n_input, n_freqs, n_channels).
        """
        raise NotImplementedError("Subclasses must implement this method. Baseline reference model the splines are applied to.")
    
    def prepare_interp_input(self, *args, **kwargs) -> tuple:
        """
        This method prepares the input for the interpolation process.
        """
        raise NotImplementedError("Subclasses must implement this method. Prepare the knots and weights for the interpolation.")
    
    def logperturbation(self, 
                        freqs: xp.ndarray, 
                        knots: xp.ndarray, 
                        weights: xp.ndarray) -> xp.ndarray:
        """
        Compute the log perturbation for the given frequencies, knots, and weights.
        """
        logperturbation = self.interp(self.xp.log10(freqs), knots, weights)

        return logperturbation.transpose(1, 2, 0)

    def __call__(self,
                 freqs: xp.ndarray,
                 args,
                 kwargs,
                 knots: xp.ndarray,
                 weights: xp.ndarray) -> jnp.ndarray:
        """
        Compute the log perturbation for the given frequencies, knots, and weights.
        """
        base = self.base_model(freqs=freqs, *args, **kwargs)
        ftol_mask = self.xp.any(self.xp.any(self.xp.abs(self.xp.diff(knots)) < self.ftol, axis=-1), axis=0)
        logperturbation = self.logperturbation(freqs=freqs, knots=knots, weights=weights)
        perturbation = 10**logperturbation
        perturbation[ftol_mask] = self.xp.nan 
        
        perturbation = jnp.asarray(perturbation)        

        out = base * perturbation
        
        return out

In [ ]:
class SplinedModelEryn(SplinedModel):
    """
    A subclass of SplinedModel compatible with Eryn's reversible jump.
    """
    def __init__(self,
                    use_gpu: bool = False,
                    ftol: float = 1e-2,
                    Ncov: int = 3, # entries in the covariance matrix, (S_aa, S_ee, S_tt)
                    Nknotsmax: int = 10,
                    leftedge: float = -4.0, # logFmin
                    rightedge: float = xp.log10(2.9e-2), # logFmax
                    interp_kwargs: dict = {}):
            
            super().__init__(use_gpu=use_gpu, ftol=ftol, interp_kwargs=interp_kwargs)

            self.Ncov = Ncov
            self.Nknotsmax = Nknotsmax
            self.leftedge = leftedge
            self.rightedge = rightedge


    def prepare_interp_input(self, args, groups, ngroups=0):

        """
        Prepares the input data for interpolation using Numba.
        Parameters:
        -----------
        args : list
            A list of arrays where the first element contains the edges and weights, 
            and the subsequent elements contain the knots. The shape of the arrays 
            determines the processing path.
        groups : list
            A list of arrays where the first element contains the group indices for 
            the edges and weights, and the subsequent elements contain the group 
            indices for the knots.
        ngroups : int, optional
            Number of groups (default is 0).
        Returns:
        --------
        sortedpositions : ndarray
            An array of sorted positions for interpolation.
        sortedweights : ndarray
            An array of sorted weights corresponding to the sorted positions.
        """

        if (args[1].shape[-1] == self.Ncov + 1) or (len(args) == 2): # this means all the weights are together or there is only one spline
            edges_weights, knots_full = self.xp.asarray(args[0]), self.xp.asarray(args[1]) #always work along the `1` axis for frequency operations # TODO may have to change this, maybe (Ncov, Nin, Nfreq) is better
            groups_knots = groups[1]           
    
            leftedge_full = edges_weights[:, 0::2]
            rightedge_full = edges_weights[:, 1::2]

            #? group_unique, group_index, group_inverse, group_count = self.xp.unique(groups_knots, return_index=True, return_counts=True, return_inverse=True)
            group_unique, group_index, group_inverse = self.xp.unique(groups_knots, return_index=True, return_counts=False, return_inverse=True)

            diff_temp = self.xp.ones_like(group_inverse)
            diff_temp[1:] = (~self.xp.diff(group_inverse).astype(bool)).astype(int)

            inds_per_group = (self.xp.cumsum(diff_temp) - 1)
            inds_group_subtract = inds_per_group[group_index][group_inverse]
            inds_per_group = inds_per_group - inds_group_subtract

            ngroups = leftedge_full.shape[0] #? (group_unique.max().item() - group_unique.min().item() ) + 1
            maxgroups = self.Nknotsmax #? group_count.max().item() if group_count.shape[0] > 0 else 0

            knots_full_nans = self.xp.full((ngroups, maxgroups, knots_full.shape[-1]), self.xp.nan)
            if leftedge_full.shape[0] != ngroups:
                breakpoint()

            leftedge_full = self.xp.concatenate((self.xp.full((ngroups,1), self.leftedge), leftedge_full), axis=1)[:, None, :]
            rightedge_full = self.xp.concatenate((self.xp.full((ngroups,1), self.rightedge), rightedge_full), axis=1)[:, None, :]

            knots_full_nans[(groups_knots, inds_per_group)] = knots_full
        
            knots_full_nans = self.xp.concatenate((leftedge_full, knots_full_nans, rightedge_full), axis = 1)

            positions = knots_full_nans[:,:,:1]
            weights = knots_full_nans[:,:,1:]

            order = self.xp.argsort(positions, axis = 1)
            sortedpositions = self.xp.take_along_axis(positions, order, axis=1)

            if (args[1].shape[-1] == self.Ncov + 1):
                sortedpositions = self.xp.repeat(sortedpositions, self.Ncov, axis = -1).transpose(2,0,1)
            else:
                sortedpositions = sortedpositions.transpose(2,0,1)

            sortedweights = self.xp.take_along_axis(weights, order, axis=1).transpose(2,0,1)

        else:
            edges_weights = self.xp.asarray(args[0])
            leftedge_full = edges_weights[:, 0::2]
            rightedge_full = edges_weights[:, 1::2]

            args_knots = [self.xp.asarray(arg) for arg in args[1:]] #still a list
            groups_knots = groups[1:] #still a list

            # groups_unique = np.unique(groups[0])
            # ngroups = (groups_unique.max().item() - groups_unique.min().item()) + 1

            ngroups = groups[0].shape[0] # RJ is not used in this branch, since it is the one handling the edges. there will be only one leaf per group

            maxgroups = self.Nknotsmax  
            
            knots_full_nans = self.xp.full((ngroups, maxgroups, 2*self.Ncov), self.xp.nan)
            leftedge_full = self.xp.concatenate((self.xp.full((ngroups, self.Ncov), self.leftedge), leftedge_full), axis=1)[:, None, :]
            rightedge_full = self.xp.concatenate((self.xp.full((ngroups, self.Ncov), self.rightedge), rightedge_full), axis=1)[:, None, :]

            for j, (arg, group) in enumerate(zip(args_knots, groups_knots)):
                group = self.xp.asarray(group)
                #? group_unique, group_index, group_inverse, group_count = self.xp.unique(group, return_index=True, return_counts=True, return_inverse=True)
                group_unique, group_index, group_inverse = self.xp.unique(group, return_index=True, return_counts=False, return_inverse=True)

                diff_temp = self.xp.ones_like(group_inverse)
                diff_temp[1:] = (~self.xp.diff(group_inverse).astype(bool)).astype(int)

                inds_per_group = (self.xp.cumsum(diff_temp) - 1)
                inds_group_subtract = inds_per_group[group_index][group_inverse]
                inds_per_group = inds_per_group - inds_group_subtract

                knots_full_nans[:,:, j][(group, inds_per_group)] = arg[:, 0]
                knots_full_nans[:,:, self.Ncov + j][(group, inds_per_group)] = arg[:, 1]

            knots_full_nans = self.xp.concatenate((leftedge_full, knots_full_nans, rightedge_full), axis = 1)

            positions = knots_full_nans[:,:,:self.Ncov]
            weights = knots_full_nans[:,:,self.Ncov:]

            order = self.xp.argsort(positions, axis = 1)
            sortedpositions = self.xp.take_along_axis(positions, order, axis=1).transpose(2,0,1)
            sortedweights = self.xp.take_along_axis(weights, order, axis=1).transpose(2,0,1)

        return sortedpositions, sortedweights